# Filtering for SACAIR runs to capture results

In [11]:
import json
from pathlib import Path

# locals
# from code.run_types import ResSet, RunRes
import pandas as pd

from src.configs import get_avail_splits
from src.que.core import CompExpInfo, Que
from src.que.shell import get_filters_crits_dropkeys, output_filtered_runs
from src.run_types import CUTOFF_9_NAMES, RESULTS_DIR


def fetch_runs(
    filters_path: Path,
    sort_keys: list[list[str]] | None = None,
    reverse: bool = False,
    top_n: int | None = None,
    output_path: str | None = None
) -> list[CompExpInfo]:
    file_filter_keys, file_criterions, file_drop_key_sets = get_filters_crits_dropkeys(
        filters_path
    )
    que = Que()
    runs = list(
        Que.list_manipulation(
            que.list_runs(
                'old_runs'
            ),
            sort_keys=sort_keys,
            reverse=reverse,
            filter_keys=file_filter_keys,
            criterions=file_criterions,
        )
    )
    
    # retrieve top n if specified
    if top_n is not None:
        runs = runs[: top_n]

    if output_path:
        output_filtered_runs(
            runs=runs,
            output_path=output_path,
            file_drop_key_sets=file_drop_key_sets,
        )

    return [CompExpInfo.model_validate(run)for run in runs] 

def load_runs(runs_path: Path) -> list[CompExpInfo]:
    with open(runs_path, "r") as f:
        return [CompExpInfo.model_validate(r) for r in json.load(f)]


In [12]:
results_dir = RESULTS_DIR / 'sacair_2026'
# runs = load_runs(results_dir / 'runs.json')
runs = fetch_runs(results_dir / 'filters.py')
print(f'{len(runs)} found')

21 found


## Now we can compare the runs

In [13]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_names = ['test', 'val']

#### Convert to DataFrame

In [14]:
df_format = [
    {
        "model": run.admin.model,
        "exp no": run.admin.exp_no,
        "run_id": run.wandb.run_id,
        "No. frames": run.data.target_length,
        "subset": run.admin.split,
    }
    | {f'{set_name} {k}': v for set_name in set_names for k, v in run.results.model_dump()[set_name][acc_type].items()}
    | {
        "best_val_acc": run.results.best_val_acc,
        "best_val_loss": run.results.best_val_loss,
        "test_loss": run.results.test.average_loss,
        "config path": run.admin.config_path,
        "weight path": run.admin.weight_path,
        }
    for run in runs
]


df = pd.DataFrame(df_format)

In [15]:
ns = [1, 5, 10]
for set_name in set_names:
    for n in ns:
        old_name, new_name = f"{set_name} top{n}", f"{set_name.capitalize()} Top-{n}"

        df = df.rename(columns={old_name: new_name})
        df[new_name] = df[new_name].apply(lambda x: f"{x * 100:.2f}")

# df

In [16]:
print(get_avail_splits())

['asl100', 'asl300_cutoff_9', 'asl2000', 'asl300', 'asl100_cutoff_9', 'asl2000_cutoff_9', 'asl1000', 'asl1000_cutoff_9']


In [21]:
subsets = CUTOFF_9_NAMES
# subsets = get_avail_splits()
for set_name in subsets:
    subdf = df[df['subset'] == set_name]
    print(f'{set_name}'.capitalize())
    # display(subdf.sort_values('Test Top-1', ascending=False))
    # display(subdf.sort_values('best_val_loss', ascending=True))
    display(subdf.sort_values('test_loss', ascending=True))

Asl100_cutoff_9


,model,exp no,run_id,No. frames,subset,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path,weight path
19,MViTv2_B_32x3,007,mi50bgp8,32,asl100_cutoff_9,77.52,93.41,96.51,78.99,94.97,96.75,80.473373,0.854901,0.919939,configfiles/asl100/MViTv2_B_32x3/exp007.toml,None
5,MViTv2_S_16x4_e,000,wifpnh92,32,asl100_cutoff_9,78.68,91.47,95.35,78.40,94.38,97.04,78.402367,0.901107,0.939804,configfiles/asl100/MViTv2_B_32x3/exp007.toml,None
20,MViTv2_B_32x3,001,079ju1dh,32,asl100_cutoff_9,73.26,92.64,95.74,77.81,94.08,97.04,81.360947,0.844280,0.969602,configfiles/asl100/MViTv2_B_32x3/exp001.toml,None
4,MViTv2_S_16x4_e,000,due3auwd,32,asl100_cutoff_9,75.97,91.47,95.74,74.56,94.08,97.34,77.810651,0.899458,0.992132,configfiles/asl100/MViTv2_B_32x3/exp007.toml,None
15,MViTv2_S_16x4,003,86cdpg62,16,asl100_cutoff_9,73.64,91.47,96.12,76.92,93.20,96.75,76.923077,0.925181,1.096831,configfiles/asl100/MViTv2_S_16x4/exp003.toml,None
0,MViTv2_S_16x4,003,qr3tw8i0,16,asl100_cutoff_9,70.16,91.86,94.96,72.49,92.90,97.04,75.147929,1.015559,1.139271,configfiles/asl100/MViTv2_S_16x4/exp003.toml,None
6,MViTv2_B_32x3_r,000,44hymv5q,16,asl100_cutoff_9,72.09,90.70,94.19,73.37,93.49,95.56,75.739645,0.971873,1.177712,configfiles/asl100/MViTv2_S_16x4/exp003.toml,None
7,MViTv2_B_32x3_r,000,xjsc2bqu,16,asl100_cutoff_9,71.71,89.15,94.19,70.41,92.60,96.15,75.147929,1.047918,1.186892,configfiles/asl100/MViTv2_S_16x4/exp003.toml,None


Asl300_cutoff_9


,model,exp no,run_id,No. frames,subset,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path,weight path
18,MViTv2_B_32x3,000,mgjsrgqn,32,asl300_cutoff_9,67.07,88.77,92.22,72.70,92.12,95.78,72.918979,1.085137,1.364747,configfiles/asl300/MViTv2_B_32x3/exp000.toml,runs/asl100_cutoff_9/MViTv2_B_32x3/exp007/chec...
14,MViTv2_S_16x4,000,v8x1dmue,16,asl300_cutoff_9,60.93,88.02,92.51,66.48,89.46,93.01,68.368479,1.292044,1.564197,configfiles/asl300/MViTv2_S_16x4/exp000.toml,runs/asl100_cutoff_9/MViTv2_S_16x4/exp003/chec...


Asl1000_cutoff_9


,model,exp no,run_id,No. frames,subset,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path,weight path
17,MViTv2_B_32x3,000,5fj94635,32,asl1000_cutoff_9,56.77,83.48,89.66,58.82,85.81,91.12,60.284605,1.781256,1.922579,configfiles/asl1000/MViTv2_B_32x3/exp000.toml,runs/asl300_cutoff_9/MViTv2_B_32x3/exp000/chec...
13,MViTv2_S_16x4,000,t2zzkxg9,16,asl1000_cutoff_9,51.12,79.37,86.19,52.95,81.54,88.66,54.808107,1.980439,2.156511,configfiles/asl1000/MViTv2_S_16x4/exp000.toml,runs/asl300_cutoff_9/MViTv2_S_16x4/exp000/chec...


Asl2000_cutoff_9


,model,exp no,run_id,No. frames,subset,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path,weight path
16,MViTv2_B_32x3,000,ap0prl54,32,asl2000_cutoff_9,44.56,76.35,83.71,44.67,77.23,84.94,47.064829,2.492716,2.548980,configfiles/asl2000/MViTv2_B_32x3/exp000.toml,runs/asl1000_cutoff_9/MViTv2_B_32x3/exp000/che...
12,MViTv2_S_16x4,000,f8r7lstq,16,asl2000_cutoff_9,39.91,70.61,79.65,40.68,72.89,81.96,42.700357,2.722687,2.827124,configfiles/asl2000/MViTv2_S_16x4/exp000.toml,runs/asl1000_cutoff_9/MViTv2_S_16x4/exp000/che...
11,MViTv2_S_16x4,001,jxuv2gmx,16,asl2000_cutoff_9,37.65,69.36,80.38,38.51,70.83,80.42,41.220010,2.784146,2.830752,configfiles/asl2000/MViTv2_S_16x4/exp001.toml,None


In [18]:


# Column used to select the best run per model/subset
SELECTION_COL = "best_val_acc"   # change to "best_val_loss" if needed

# Optional: if you want custom model names with \cite, set escape=False below
# and store the LaTeX string directly in the 'model' column.
# ==========================================================

def make_latex_table_for_subset(df, subset):
    """Select best run per model and return LaTeX table for one subset."""
    sub = df[df["subset"] == subset].copy()
    sub["model"] = sub["model"].astype(str).str.strip()

    # Convert selection column to numeric and drop rows with missing value
    sub[SELECTION_COL] = pd.to_numeric(sub[SELECTION_COL], errors="coerce")
    sub = sub.dropna(subset=[SELECTION_COL])

    # Select best run: highest accuracy or lowest loss
    if SELECTION_COL == "best_val_loss":
        idx = sub.groupby("model")[SELECTION_COL].idxmin()
    else:
        idx = sub.groupby("model")[SELECTION_COL].idxmax()
    best = sub.loc[idx].copy()

    # Keep only needed columns and rename
    best = best[["model", "Test Top-1", "Test Top-5", "Test Top-10"]].rename(
        columns={
            "model": "Model",
            "Test Top-1": "Acc@1",
            "Test Top-5": "Acc@5",
            "Test Top-10": "Acc@10",
        }
    )

    # Ensure metrics are numeric (to_latex will format them)
    for col in ["Acc@1", "Acc@5", "Acc@10"]:
        best[col] = pd.to_numeric(best[col], errors="coerce")

    # Generate LaTeX using .to_latex()
    latex = best.to_latex(
        index=False,
        na_rep="-",
        float_format="%.2f",
        caption=f"{subset} Results",
        label=f"tab:{subset.lower().replace('-', '_')}",
        position="htbp",
    )
    return latex


# Generate and print each table
for subset in subsets:
    print(f"% ===== Table for {subset} =====")
    print(make_latex_table_for_subset(df, subset))
    print("\n")   # blank line between tables

% ===== Table for asl100_cutoff_9 =====
\begin{table}[htbp]
\caption{asl100_cutoff_9 Results}
\label{tab:asl100_cutoff_9}
\begin{tabular}{lrrr}
\toprule
Model & Acc@1 & Acc@5 & Acc@10 \\
\midrule
MViTv2_B_32x3 & 73.26 & 92.64 & 95.74 \\
MViTv2_B_32x3_r & 72.09 & 90.70 & 94.19 \\
MViTv2_S_16x4 & 73.64 & 91.47 & 96.12 \\
MViTv2_S_16x4_e & 78.68 & 91.47 & 95.35 \\
\bottomrule
\end{tabular}
\end{table}



% ===== Table for asl300_cutoff_9 =====
\begin{table}[htbp]
\caption{asl300_cutoff_9 Results}
\label{tab:asl300_cutoff_9}
\begin{tabular}{lrrr}
\toprule
Model & Acc@1 & Acc@5 & Acc@10 \\
\midrule
MViTv2_B_32x3 & 67.07 & 88.77 & 92.22 \\
MViTv2_S_16x4 & 60.93 & 88.02 & 92.51 \\
\bottomrule
\end{tabular}
\end{table}



% ===== Table for asl1000_cutoff_9 =====
\begin{table}[htbp]
\caption{asl1000_cutoff_9 Results}
\label{tab:asl1000_cutoff_9}
\begin{tabular}{lrrr}
\toprule
Model & Acc@1 & Acc@5 & Acc@10 \\
\midrule
MViTv2_B_32x3 & 56.77 & 83.48 & 89.66 \\
MViTv2_S_16x4 & 51.12 & 79.37 & 86.1

In [19]:
print("Unique subset values:", df["subset"].unique())
print("Unique model values:", df["model"].unique())
print("Columns:", df.columns.tolist())

Unique subset values: ['asl100_cutoff_9' 'asl100' 'asl2000_cutoff_9' 'asl1000_cutoff_9'
 'asl300_cutoff_9']
Unique model values: ['MViTv2_S_16x4' 'MViTv2_B_32x3_r' 'MViTv2_S_16x4_e' 'MViTv2_B_32x3']
Columns: ['model', 'exp no', 'run_id', 'No. frames', 'subset', 'Test Top-1', 'Test Top-5', 'Test Top-10', 'Val Top-1', 'Val Top-5', 'Val Top-10', 'best_val_acc', 'best_val_loss', 'test_loss', 'config path', 'weight path']
